# Notebook 02 — Pathway enrichment

For each selected component, run KEGG pathway enrichment over the PIP-significant RNA and methylation features. RNA features are already gene symbols, tested against all 25,098 genes of the RNA input. Methylation features are mapped to genes via the array manifest's `RefGene` column (every gene listed for each CpG) and tested against Enrichr's default background, which Data S1 used only for Component 74; the other CpG terms in Data S1 used the 19,872 `RefGene` genes of the array (Methods), so their P-values differ from these. No protein-derived results are reported (Methods).

**ChIP is not included in default enrichment.** The deposited H3K9ac peak manifest (`H3K9acDomains.csv`) carries chromosome coordinates and a Promoter/Enhancer/Other classification, but no gene-symbol or nearest-gene column. Mapping ChIP peaks to genes requires an external nearest-gene step (e.g., `bedtools closest`, or HOMER's `annotatePeaks.pl`), which is out of scope for these notebooks. The manuscript reports no pathway enrichment for H3K9ac peaks.

Returns a results dictionary `{component_id: {omic: DataFrame}}` you can flatten or save however you want.

## Required Synapse files

Download version 2 of Synapse dataset `syn75275226` (AD Knowledge Portal). Place all files in a
single directory and set `DEPOSIT_DIR` below. Component IDs are 1–519 and 521–548
(ID 520 is retired), so select components by ID, not by row number.

| File | Shape | Description |
|---|---|---|
| `samples_final.csv` | 547 × 276 | Component × sample loadings |
| `cell_type.csv` | 547 × 6 | Component × cell-type loadings (Ast, Oli, Inh, OPC, Exc, Mic) |
| `RNA.txt` | 25,098 × 547 | Gene × component loadings (PIP > 0.5 masked) |
| `meth.txt` | 51,798 × 547 | CpG × component loadings (PIP > 0.5 masked) |
| `chip.txt` | 26,384 × 547 | H3K9ac peak × component loadings (PIP > 0.5 masked) |

## Required reference annotations

Not bundled with this repository: download the three tables into `references/` (sources and steps in `references/README.md`). The gene annotation is extracted from the Ensembl release 100 GTF; the array manifest and the peak coordinates come with the ROSMAP DNA methylation and H3K9ac ChIP-seq data on the AD Knowledge Portal. All four notebooks load them with the deposit.

| File | Description |
|---|---|
| `Homo_sapiens.GRCh38.100.txt` | Gene annotation (Ensembl 100) |
| `ROSMAP_arrayMethylation_metaData.tsv` | Illumina methylation array manifest |
| `H3K9acDomains.csv` | H3K9ac ChIP-seq peak coordinates (Klein et al.) |

Clinical metadata (`ROSMAP_clinical.csv`) is **not redistributed** — it is
controlled-access (request via the
[AD Knowledge Portal DUC](https://adknowledgeportal.synapse.org/Data%20Access)). Notebooks that need
clinical variables document their dependency inline and degrade gracefully
when the file is absent.


## Setup

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add the repo root to sys.path so we can import the helper module.
sys.path.insert(0, "..")
import g_ros_release as gr

gr.apply_plot_style()


In [2]:
# Path configuration — edit these two paths to match your local layout.
# `DEPOSIT_DIR` is where you downloaded the Synapse files (dataset
# syn75275226, version 2). `REFERENCES_DIR` is the `references/`
# directory in this repo, where you put the reference tables.

# DEPOSIT_DIR is auto-resolved between two common layouts:
#   1. `synapse/` inside the cloned repo (default — what end-users get when
#      they download the deposit into a `synapse/` folder at the repo root)
#   2. `synapse/` as a sibling of the repo (the local-prep layout used to
#      keep code and data physically separate during preparation)
# To override either default, set the env var G_ROS_DEPOSIT_DIR.
import os as _os
DEPOSIT_DIR = _os.environ.get("G_ROS_DEPOSIT_DIR", "../synapse")
if not _os.path.exists(_os.path.join(DEPOSIT_DIR, "samples_final.csv")):
    _fallback = "../../synapse"
    if _os.path.exists(_os.path.join(_fallback, "samples_final.csv")):
        DEPOSIT_DIR = _fallback
        print(f"[notebook] using sibling deposit layout: {DEPOSIT_DIR}")
REFERENCES_DIR = _os.environ.get("G_ROS_REFERENCES_DIR", "../references")   # see references/README.md

deposit = gr.load_deposit(DEPOSIT_DIR, REFERENCES_DIR, load_references=True)
print(f"Loaded {len(deposit.samples)} components × {deposit.samples.shape[1]} samples")


Loaded 547 components × 276 samples


## Configuration

In [3]:
# Choose which components to run. Defaults to the nine components shown
# in the manuscript figures. To sweep all 547 components, replace with:
#     COMPONENTS = list(deposit.samples.index)   # IDs 1–519 and 521–548
# To run a single component, e.g. just Component 74:
#     COMPONENTS = [74]
#
# Components and their manuscript figure (also in gr.HEADLINE_COMPONENTS):
#    74: Figure 2 (sex-associated regulatory variation)
#   278: Figure 3 (sex-biased component shared by OPCs and oligodendrocytes)
#   172: Figure 4 (neuron-dominated; neuritic plaque burden)
#   277: Figure 5 (neuron-dominated; clinical Alzheimer's dementia)
#    21: Sup Fig S7 (olfactory receptor expression and its cis-genetic regulation)
#   351: Sup Fig S8 (single-individual multi-omic profile)
#     2: Sup Fig S9 (single-individual multi-omic profile)
#    87: Sup Fig S10 (single-individual multi-omic profile)
#   177: Sup Fig S11 (dominated by one donor)

COMPONENTS = [74, 278, 172, 277, 21, 351, 2, 87, 177]

# Omics to enrich. KEGG operates on gene symbols; the helper
# `gr.features_to_genes` maps each omic's features to gene symbols
# (RNA: identity; meth: every RefGene gene of the array manifest). ChIP is
# omitted; see the notebook header for the reason.
OMICS = ['RNA', 'meth']

# gseapy enrichr cutoff (Adjusted P-value)
CUTOFF = 0.25

# Where to save enrichment tables. Set to None to skip saving.
SAVE_DIR = None  # e.g. "./out/kegg"
if SAVE_DIR is not None:
    os.makedirs(SAVE_DIR, exist_ok=True)


## Run enrichment

Each call to `gr.kegg_enrichment` makes a network request to the Enrichr API via gseapy. A full sweep of 547 components × 2 omics is ~1,094 requests; expect to be rate-limited and to take 30+ minutes. For the default (9 components), this runs in roughly a minute.

In [4]:
results: dict[int, dict[str, pd.DataFrame]] = {}

for cid in COMPONENTS:
    if not deposit.has_component(cid):
        print(f'Component {cid} not in deposit; skipping.')
        continue
    results[cid] = {}
    for omic in OMICS:
        gene_list = gr.features_to_genes(deposit, cid, omic)
        if not gene_list:
            print(f'  Component {cid} / {omic}: no PIP-significant features → skip')
            results[cid][omic] = pd.DataFrame()
            continue
        try:
            enr = gr.kegg_enrichment(deposit, cid, omic,
                                    gene_list=gene_list, cutoff=CUTOFF)
        except Exception as e:
            print(f'  Component {cid} / {omic}: enrichr call failed ({e}); skip')
            enr = pd.DataFrame()
        results[cid][omic] = enr
        n_terms = 0 if enr.empty else (enr['Adjusted P-value'] < CUTOFF).sum()
        print(f'  Component {cid} / {omic}: {len(gene_list)} genes → {n_terms} term(s) at adj.P < {CUTOFF}')

        if SAVE_DIR is not None and not enr.empty:
            out_path = os.path.join(SAVE_DIR, f'comp{cid}_{omic}_kegg.csv')
            enr.to_csv(out_path, index=False)


  Component 74 / RNA: 161 genes → 0 term(s) at adj.P < 0.25
  Component 74 / meth: 1685 genes → 77 term(s) at adj.P < 0.25
  Component 278 / RNA: 1114 genes → 177 term(s) at adj.P < 0.25
  Component 278 / meth: no PIP-significant features → skip
  Component 172 / RNA: 1013 genes → 130 term(s) at adj.P < 0.25
  Component 172 / meth: no PIP-significant features → skip
  Component 277 / RNA: 1180 genes → 131 term(s) at adj.P < 0.25
  Component 277 / meth: no PIP-significant features → skip
  Component 21 / RNA: 555 genes → 24 term(s) at adj.P < 0.25
  Component 21 / meth: no PIP-significant features → skip
  Component 351 / RNA: 707 genes → 4 term(s) at adj.P < 0.25
  Component 351 / meth: 5 genes → 101 term(s) at adj.P < 0.25
  Component 2 / RNA: 277 genes → 0 term(s) at adj.P < 0.25
  Component 2 / meth: 5 genes → 7 term(s) at adj.P < 0.25
  Component 87 / RNA: 255 genes → 0 term(s) at adj.P < 0.25
  Component 87 / meth: 3 genes → 35 term(s) at adj.P < 0.25
  Component 177 / RNA: 155 ge

## Inspect results — top terms per component, per omic

Display the top 5 enriched KEGG terms per (component, omic). Empty entries mean no term passed the cutoff.

In [5]:
for cid, by_omic in results.items():
    print(f'\n=== Component {cid} — {gr.HEADLINE_COMPONENTS.get(cid, "non-headline")} ===')
    for omic, df in by_omic.items():
        if df.empty:
            print(f'  [{omic}] no enriched terms')
            continue
        top = df.head(5)[['Term', 'Adjusted P-value', 'Combined Score']]
        print(f'  [{omic}] top {len(top)} of {len(df)}:')
        for _, row in top.iterrows():
            print(f"    {row['Term']}  (adj.P={row['Adjusted P-value']:.2e}, CS={row['Combined Score']:.1f})")



=== Component 74 — Figure 2: sex-associated regulatory variation ===
  [RNA] top 5 of 163:
    Cell adhesion molecules  (adj.P=6.45e-01, CS=24.1)
    Glioma  (adj.P=6.45e-01, CS=2.2)
    Human cytomegalovirus infection  (adj.P=6.45e-01, CS=1.6)
    Cytokine-cytokine receptor interaction  (adj.P=6.45e-01, CS=1.6)
    Focal adhesion  (adj.P=6.45e-01, CS=1.7)
  [meth] top 5 of 303:
    Adrenergic signaling in cardiomyocytes  (adj.P=2.36e-04, CS=42.0)
    Signaling pathways regulating pluripotency of stem cells  (adj.P=9.73e-04, CS=33.1)
    Pathways in cancer  (adj.P=9.73e-04, CS=20.4)
    Cushing syndrome  (adj.P=9.73e-04, CS=29.6)
    Growth hormone synthesis, secretion and action  (adj.P=9.73e-04, CS=32.2)

=== Component 278 — Figure 3: sex-biased component shared by OPCs and mature oligodendrocytes ===
  [RNA] top 5 of 286:
    Ras signaling pathway  (adj.P=2.99e-07, CS=76.9)
    MAPK signaling pathway  (adj.P=2.99e-07, CS=66.2)
    AMPK signaling pathway  (adj.P=2.99e-07, CS=103.9)


## Optional: flatten to a single long-format table

Useful for exporting a single CSV that anyone can filter, or for joining to `Data_S1.csv`.

In [6]:
rows = []
for cid, by_omic in results.items():
    for omic, df in by_omic.items():
        if df.empty:
            continue
        for _, r in df.iterrows():
            rows.append({
                'component_id': cid, 'omic': omic,
                'term': r['Term'],
                'adjusted_p': r['Adjusted P-value'],
                'combined_score': r['Combined Score'],
                'overlap': r.get('Overlap'),
                'genes': r.get('Genes'),
            })
long = pd.DataFrame(rows)
print(f'Long-format enrichment table: {long.shape}')
if not long.empty:
    display(long.head(20))


Long-format enrichment table: (2168, 7)


,component_id,omic,term,adjusted_p,combined_score,overlap,genes
0,74,RNA,Cell adhesion molecules,0.645385,24.121301,NaN,NLGN4Y;CD40;NLGN4X;CD99
1,74,RNA,Glioma,0.645385,2.224524,NaN,AKT1
2,74,RNA,Human cytomegalovirus infection,0.645385,1.626405,NaN,AKT1;GNAI2
3,74,RNA,Cytokine-cytokine receptor interaction,0.645385,1.645155,NaN,CD40;IL17RA
4,74,RNA,Focal adhesion,0.645385,1.664180,NaN,COL4A5;AKT1
5,74,RNA,Melanoma,0.645385,2.468332,NaN,AKT1
6,74,RNA,Gastric acid secretion,0.645385,2.468332,NaN,GNAI2
7,74,RNA,Renal cell carcinoma,0.645385,2.535252,NaN,AKT1
8,74,RNA,Complement and coagulation cascades,0.645385,2.535252,NaN,KLKB1
9,74,RNA,Coronavirus disease,0.645385,1.784415,NaN,RPS4X;RPS4Y1
